In [ ]:
# Run this notebooks with chemgifs conda env

In [5]:
!pip install pandas

In [37]:
import pandas as pd
from tqdm import tqdm
import os
import numpy as np
import tarfile
from collections import Counter

In [7]:
# Define some paths
root = '../../../Documents_GPU/mtb-targeted-protein-degradation/scripts'
PATH_TO_DOCKING_RESULTS_ORIGINAL = os.path.join(root, "..", "processed", "unidock_docking", 'docking_results')
PATH_TO_DOCKING_RESULTS_REAL = os.path.join(root, "..", "processed", "unidock_REAL_docking", 'docking_results')
PATH_TO_INPUT_LIGANDS = os.path.join(root, "..", "processed", "unidock_REAL_docking", 'input_ligands')

# Load pocket detection data
pocket_detection_data = pd.read_csv(os.path.join(root, "..", "processed", "pocket_detection_data.csv"))

In [8]:
DOCKING_RESULTS_ORIGINAL = {}
DOCKING_RESULTS_REAL = {}
DOCKING_RESULTS_REAL_BACKGROUND = {}

# For each pocket
for pocket in tqdm(sorted(os.listdir(PATH_TO_DOCKING_RESULTS_ORIGINAL))):
    scores = pd.read_csv(os.path.join(PATH_TO_DOCKING_RESULTS_ORIGINAL, pocket, 'report.csv'))
    DOCKING_RESULTS_ORIGINAL[pocket] = {i: j for i, j in zip(scores['compound'], scores['score'])}

# For each pocket
for pocket in tqdm(sorted(os.listdir(PATH_TO_DOCKING_RESULTS_REAL))):
    # try:
    lines = open(os.path.join(PATH_TO_INPUT_LIGANDS, f"input_ligands_{pocket}.txt"), "r").readlines()
    lines = [i.strip().replace(".sdf", "").split("/")[-1] for i in lines]
    actives = set(lines[:100000])
    inactives = set(lines[100000:])
    scores = pd.read_csv(os.path.join(PATH_TO_DOCKING_RESULTS_REAL, pocket, 'report.csv'))
    scores['set'] = ["inactive" if i in inactives else "active" for i in scores['compound']]
    scores_actives = scores[scores['set'] == 'active'].reset_index(drop=True)
    scores_inactives = scores[scores['set'] == 'inactive'].reset_index(drop=True)
    DOCKING_RESULTS_REAL[pocket] = {i: j for i, j in zip(scores_actives['compound'], scores_actives['score'])}
    DOCKING_RESULTS_REAL_BACKGROUND[pocket] = {i: j for i, j in zip(scores_inactives['compound'], scores_inactives['score'])}
    # except:
    #     pass

100%|██████████| 276/276 [00:34<00:00,  8.10it/s]


In [34]:
### SELECT TOP MOLECULES AND PREPARE GIF USING CHEMGIFS ###

In [20]:
N = 10_000
ACTIVES = []
proteins = set(pocket_detection_data['Uniprot AC'])
ACTIVES_PER_PROTEIN = {i: set() for i in proteins}
for pocket in tqdm(sorted(DOCKING_RESULTS_REAL)):
    act = sorted(DOCKING_RESULTS_REAL[pocket], key = lambda x: DOCKING_RESULTS_REAL[pocket][x])[:N]
    ACTIVES.extend(act)
    ACTIVES_PER_PROTEIN[pocket.split("_")[1]].update(set(act))


# Get multi-target molecules
counts = Counter(ACTIVES)
counts_proteins = Counter([cpd for protein in proteins for cpd in ACTIVES_PER_PROTEIN[protein]])
active_21_proteins = [cmpd for cmpd, c in counts_proteins.items() if c >= 21]
print(f"TOP-{N} actives")
print(f"Compounds that are active at least once: {len(set(ACTIVES))}")
print(f"Compounds that are active in at least 21 proteins: {len(active_21_proteins)}")

# Get ID to SMILES mapping
ID_TO_SMILES = pd.read_csv(os.path.join(root, "..", "processed", "enamine_REAL_characterization", "enamine_REAL.tsv"), sep='\t')
ID_TO_SMILES = {i: j for i,j in zip(ID_TO_SMILES['id'], ID_TO_SMILES['smiles'])}
SMILES = [ID_TO_SMILES[i] for i in active_21_proteins]

100%|██████████| 276/276 [00:04<00:00, 63.85it/s]


TOP-10000 actives
Compounds that are active at least once: 620557
Compounds that are active in at least 21 proteins: 398


In [ ]:
with open(os.path.join(root, "..", "processed", "unidock_REAL_docking", "multi_target_actives_smiles.smi"), "w") as f:
    for smi in SMILES:
        f.write(smi + "\n")

In [ ]:
!chemgifs -i ../processed/unidock_REAL_docking/multi_target_actives_smiles.smi -o \
    ../processed/unidock_REAL_docking/multi_target_actives_smiles.gif -c yellow -s 512 -d 200

0it [00:00, ?it/s]/bin/sh: 1: /home/acomajuncosa/miniconda3/envs/chemgifs/lib/python3.12/site-packages/tools/linux_x86/mol2svg: not found
0it [00:00, ?it/s]
Traceback (most recent call last):
  File "/home/acomajuncosa/miniconda3/envs/chemgifs/bin/chemgifs", line 7, in <module>
    sys.exit(main())
             ^^^^^^
  File "/home/acomajuncosa/miniconda3/envs/chemgifs/lib/python3.12/site-packages/chemgifs/main.py", line 191, in main
    run(
  File "/home/acomajuncosa/miniconda3/envs/chemgifs/lib/python3.12/site-packages/chemgifs/main.py", line 159, in run
    get_mol_svg(name, smiles, tmp_dir, color_name)
  File "/home/acomajuncosa/miniconda3/envs/chemgifs/lib/python3.12/site-packages/chemgifs/main.py", line 112, in get_mol_svg
    get_raw_mol_svg(name, smiles, output_dir, color)
  File "/home/acomajuncosa/miniconda3/envs/chemgifs/lib/python3.12/site-packages/chemgifs/main.py", line 70, in get_raw_mol_svg
    subprocess.run(cmd, shell=True, check=True)
  File "/home/acomajuncosa/mini

In [ ]:
### GET TOP POSES PER POCKET ###

In [54]:
for pocket in sorted(DOCKING_RESULTS_REAL):

    # Get top molecules
    top_mols = DOCKING_RESULTS_REAL[pocket]
    top_mols = sorted(top_mols, key = lambda x: top_mols[x])[:6]

    # Read tar file and extract top poses
    with tarfile.open(os.path.join(PATH_TO_DOCKING_RESULTS_REAL, pocket, 'docking.tar.gz'), 'r:gz') as tar:

        # Create top poses directory
        os.makedirs(os.path.join(PATH_TO_DOCKING_RESULTS_REAL, pocket, 'top_poses'), exist_ok=True)

        # Get top molecules
        for top_mol in top_mols:
            file = tar.extractfile(f"docking/{top_mol}_out.sdf").read()
            with open(os.path.join(PATH_TO_DOCKING_RESULTS_REAL, pocket, 'top_poses', f"{top_mol}_out.sdf"), "wb") as f:
                f.write(file)

    break

In [55]:
top_mols

['s_27____20046792____12360382',
 's_2430____22523358____8738422',
 's_68____27751140____21690634',
 's_27____7941186____61078',
 'm_12____6286504____25386190____25384538',
 's_22____13154140____28465902']

In [32]:
DOCKING_RESULTS_REAL['alphafold2_P9WFS9_model_0_pocket_1']["s_27____20046792____12360382"]

-11.799

In [33]:
PATH_TO_DOCKING_RESULTS = os.path.join(root, "..", "processed", "unidock_REAL_docking", "docking_results")